# Jigsaw · Rule and example baseline

Self-contained offline CPU inference. Add the official competition dataset, disable internet, then **Save Version → Save & Run All**. The official runtime limit is 12 hours. The completed notebook writes `/kaggle/working/submission.csv`.

This is a reference baseline; no medal-level result is claimed. The competition ended October 23, 2025. A late-submission button was visible but disabled while signed out on September 7, 2026; authenticated eligibility is unverified.

Source: https://www.kaggle.com/competitions/jigsaw-agile-community-rules/overview

In [ ]:
from __future__ import annotations
import os
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'
import hashlib
import json
import threading
import time
from datetime import UTC, datetime
from typing import Any
from pathlib import Path
import importlib.metadata


## Canonical validated schema and model
The following cells are generated directly from the repository's schema and model modules. CI checks that they match the package.

In [ ]:
"""Competition schemas and deterministic synthetic data for software verification."""


import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

EXAMPLES = ["positive_example_1", "positive_example_2", "negative_example_1", "negative_example_2"]
TEXT = ["body", "rule", "subreddit", *EXAMPLES]
FILES = ["train.csv", "test.csv", "sample_submission.csv"]


def normalize(text: str) -> str:
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", text)).strip().casefold()


def validate_frame(df: pd.DataFrame, *, train: bool) -> None:
    required = ["row_id", *TEXT] + (["rule_violation"] if train else [])
    missing = set(required) - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")
    if df.empty or df.row_id.isna().any() or df.row_id.duplicated().any():
        raise ValueError("Rows and unique, non-null row_id values are required")
    for column in TEXT:
        if not df[column].map(lambda v: isinstance(v, str) and bool(v.strip())).all():
            raise ValueError(f"Empty or non-text values in {column}")
    if train and (df.rule_violation.isna().any() or not df.rule_violation.isin([0, 1]).all()):
        raise ValueError("Targets must be binary 0/1 with no missing values")
    if not train and "rule_violation" in df:
        raise ValueError("Test data must not contain target labels")


def validate_submission(submission: pd.DataFrame, sample: pd.DataFrame) -> None:
    if list(submission.columns) != ["row_id", "rule_violation"]:
        raise ValueError("Submission must have exactly row_id,rule_violation")
    if submission.empty or submission.row_id.isna().any() or submission.row_id.duplicated().any():
        raise ValueError("Submission IDs must be unique and non-null")
    if not submission.row_id.equals(sample.row_id):
        raise ValueError("Submission row IDs or order differ from sample")
    p = submission.rule_violation.to_numpy(dtype=float)
    if not np.isfinite(p).all() or ((p < 0) | (p > 1)).any():
        raise ValueError("Predictions must be finite probabilities in [0,1]")


def load_data(directory: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    for name in FILES:
        if not (directory / name).is_file():
            raise FileNotFoundError(f"Missing {directory / name}; run jigsaw download first")
    train, test, sample = (pd.read_csv(directory / name) for name in FILES)
    validate_frame(train, train=True)
    validate_frame(test, train=False)
    if list(sample.columns) != ["row_id", "rule_violation"]:
        raise ValueError("Unexpected sample submission columns")
    if not test.row_id.equals(sample.row_id):
        raise ValueError("Test and sample submission IDs/order differ")
    validate_submission(sample, sample)
    if set(train.row_id) & set(test.row_id):
        raise ValueError("Train and test row IDs overlap")
    return train, test, sample


def audit(train: pd.DataFrame, test: pd.DataFrame) -> dict:
    train_bodies = set(train.body.map(normalize))
    test_bodies = set(test.body.map(normalize))
    return {
        "train_rows": len(train),
        "preview_test_rows": len(test),
        "train_rules": sorted(train.rule.unique().tolist()),
        "preview_test_rules": sorted(test.rule.unique().tolist()),
        "duplicate_training_bodies": int(train.body.map(normalize).duplicated().sum()),
        "train_test_body_overlap": len(train_bodies & test_bodies),
        "label_prevalence": float(train.rule_violation.mean()),
        "by_rule": train.groupby("rule")
        .rule_violation.agg(["size", "mean"])
        .reset_index()
        .to_dict("records"),
        "test_note": "Downloaded test is a preview; hidden evaluation can replace it.",
    }


def synthetic(directory: Path) -> None:
    """Tiny authored examples; never use these metrics as competition performance."""
    directory.mkdir(parents=True, exist_ok=True)
    if any((directory / f).exists() for f in FILES):
        raise FileExistsError("Synthetic generation refuses to overwrite existing data")
    rows = []
    for r, rule in enumerate(["No advertisements", "No personal insults"]):
        for i in range(48):
            label = i % 2
            phrase = (
                ["buy discount offer", "you stupid fool"][r]
                if label
                else "thoughtful topic discussion"
            )
            rows.append(
                {
                    "row_id": r * 100 + i,
                    "body": f"{phrase} uniqueitem{r}x{i}",
                    "rule": rule,
                    "subreddit": f"community{i % 4}",
                    "positive_example_1": "buy discount coupon" if r == 0 else "you foolish idiot",
                    "positive_example_2": "sale offer now" if r == 0 else "stupid personal insult",
                    "negative_example_1": "thank you for this thoughtful discussion",
                    "negative_example_2": "interesting topic worth discussing",
                    "rule_violation": label,
                }
            )
    train = pd.DataFrame(rows)
    test = train.iloc[:8].drop(columns="rule_violation").copy()
    test["row_id"] = np.arange(1000, 1008)
    test["body"] = test.body + " unseen"
    sample = pd.DataFrame({"row_id": test.row_id, "rule_violation": 0.5})
    for frame, name in zip([train, test, sample], FILES, strict=True):
        frame.to_csv(directory / name, index=False)
    (directory / "SYNTHETIC.txt").write_text("SOFTWARE TEST DATA. NOT COMPETITION RESULTS.\n")


In [ ]:
"""CPU reference models with explicit comment/example interactions."""


import warnings

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression



class LexicalClassifier:
    """A transparent reference point; this model is not claimed to understand new policies."""

    def __init__(self, context: bool = True, seed: int = 2025):
        self.context = context
        self.seed = seed
        self.vectorizer = TfidfVectorizer(
            ngram_range=(1, 2), sublinear_tf=True, max_features=40000, dtype=np.float64
        )
        self.classifier = LogisticRegression(
            C=2.0, solver="liblinear", max_iter=2000, random_state=seed
        )

    def _features(self, frame: pd.DataFrame):
        body = self.vectorizer.transform(frame.body)
        if not self.context:
            return body
        sims = [
            np.asarray(body.multiply(self.vectorizer.transform(frame[c])).sum(axis=1)).ravel()
            for c in ["rule", *EXAMPLES]
        ]
        pos = np.maximum(sims[1], sims[2])
        neg = np.maximum(sims[3], sims[4])
        extra = np.column_stack([*sims, pos, neg, pos - neg])
        return hstack([body, csr_matrix(extra)], format="csr")

    def fit(self, frame: pd.DataFrame) -> LexicalClassifier:
        columns = ["body", "rule", *EXAMPLES] if self.context else ["body"]
        corpus = [text for column in columns for text in frame[column]]
        self.vectorizer.fit(corpus)
        with warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)
            self.classifier.fit(self._features(frame), frame.rule_violation)
        return self

    def predict(self, frame: pd.DataFrame) -> np.ndarray:
        return self.classifier.predict_proba(self._features(frame))[:, 1]

    def coefficients(self) -> pd.DataFrame:
        names = self.vectorizer.get_feature_names_out().tolist()
        if self.context:
            names += [
                "similarity_rule",
                *[f"similarity_{c}" for c in EXAMPLES],
                "max_positive_similarity",
                "max_negative_similarity",
                "similarity_margin",
            ]
        return pd.DataFrame({"feature": names, "coefficient": self.classifier.coef_[0]})


In [ ]:
class Progress:
    """Every long operation emits start, heartbeat, finish/failure, and elapsed seconds."""

    _context = threading.local()

    def __init__(
        self,
        path: Path,
        stage: str,
        heartbeat_seconds: float = 15,
        *,
        total_started: float | None = None,
    ):
        self.path = path
        self.stage = stage
        self.interval = heartbeat_seconds
        self.started = time.monotonic()
        self.total_started = self.started if total_started is None else total_started
        self.explicit_total_started = total_started
        self.stop = threading.Event()
        self.guard = threading.Lock()
        self.thread: threading.Thread | None = None

    def emit(self, event: str, **fields: Any) -> None:
        record = {
            "timestamp": datetime.now(UTC).isoformat(timespec="seconds"),
            "stage": self.stage,
            "event": event,
            "elapsed_seconds": round(time.monotonic() - self.started, 3),
            "stage_elapsed_seconds": round(time.monotonic() - self.started, 3),
            "total_elapsed_seconds": round(time.monotonic() - self.total_started, 3),
            **fields,
        }
        line = json.dumps(record, allow_nan=False)
        with self.guard:
            self.path.parent.mkdir(parents=True, exist_ok=True)
            with self.path.open("a", encoding="utf-8") as stream:
                stream.write(line + "\n")
                stream.flush()
            print(line, flush=True)

    def _heartbeat(self) -> None:
        while not self.stop.wait(self.interval):
            self.emit("heartbeat")

    def __enter__(self) -> Progress:
        self.previous_total_started = getattr(self._context, "started", None)
        inherited = self.previous_total_started
        if self.explicit_total_started is None and inherited is not None:
            self.total_started = inherited
        self._context.started = self.total_started
        self.emit("started")
        self.thread = threading.Thread(target=self._heartbeat, daemon=True)
        self.thread.start()
        return self

    def __exit__(self, kind, error, traceback) -> None:
        self.stop.set()
        if self.thread:
            self.thread.join(timeout=2)
        try:
            self.emit(
                "failed" if error else "completed", error_type=kind.__name__ if kind else None
            )
        finally:
            self._context.started = self.previous_total_started

In [ ]:
input_root = Path(os.environ.get("JIGSAW_KAGGLE_INPUT", "/kaggle/input/jigsaw-agile-community-rules"))
output_root = Path(os.environ.get("JIGSAW_KAGGLE_OUTPUT", "/kaggle/working"))
output_root.mkdir(parents=True, exist_ok=True)
with Progress(output_root / "events.jsonl", "offline_submission") as log:
    train, test, sample = load_data(input_root)
    log.emit("data_validated", train_rows=len(train), test_rows=len(test))
    model = LexicalClassifier(context=True).fit(train)
    log.emit("model_fitted")
    pieces = []
    for offset in range(0, len(test), 5000):
        pieces.append(model.predict(test.iloc[offset:offset + 5000]))
        log.emit("prediction_batch", completed_rows=min(offset + 5000, len(test)), total_rows=len(test))
    submission = pd.DataFrame({"row_id": test.row_id, "rule_violation": np.concatenate(pieces)})
    validate_submission(submission, sample)
    temporary = output_root / "submission.csv.partial"
    submission.to_csv(temporary, index=False)
    os.replace(temporary, output_root / "submission.csv")
    manifest = {
        "rows": len(submission),
        "synthetic": (input_root / "SYNTHETIC.txt").exists(),
        "input_hashes": {name: hashlib.sha256((input_root / name).read_bytes()).hexdigest() for name in FILES},
        "packages": {name: importlib.metadata.version(name) for name in ["numpy", "pandas", "scipy", "scikit-learn"]},
    }
    (output_root / "submission_manifest.json").write_text(json.dumps(manifest, indent=2))
    log.emit("SUBMISSION_VALIDATED", rows=len(submission))
submission.head()